# Phase 1: Retrieval Basics

This notebook covers the Phase-1 objectives:
- Implement and compare **TF-IDF**, **BM25+**, and **Embedding-based** retrieval.
- Evaluate with standard IR metrics: **Precision@K**, **Recall@K**, **MRR@K**, **MAP@K**.

The dense model is optional at runtime because it is slower.


In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError("Could not locate the project root containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_data, build_qrels_lookup
from src.preprocess import create_content_column
from src.models import run_tfidf_search, run_bm25_search, run_dense_search
from src.evaluation import evaluate_retrieval


In [2]:
import sys
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError("Could not locate the project root containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_data, build_qrels_lookup
from src.preprocess import create_content_column

DATA_DIR = PROJECT_ROOT / 'data'
EVAL_K = 10

# Subset controls to keep notebook practical.
DOC_LIMIT = 10000
QUERY_LIMIT = 200

RUN_DENSE = True
DENSE_QUERY_LIMIT = 50

docs_df, train_queries_df, _, qrels_raw = load_data(DATA_DIR)
qrels = build_qrels_lookup(qrels_raw)

docs_df = create_content_column(docs_df, ['title', 'text', 'tags'])
train_queries_df = create_content_column(train_queries_df, ['title', 'text'])

docs_eval = docs_df.head(DOC_LIMIT).copy()
queries_eval = train_queries_df.head(QUERY_LIMIT).copy()

print('docs_eval:', docs_eval.shape)
print('queries_eval:', queries_eval.shape)
print('EVAL_K:', EVAL_K)


docs_eval: (10000, 6)
queries_eval: (200, 6)
EVAL_K: 10


In [3]:
def evaluate_model(name, retrieval_results, qrels_lookup, k):
    metrics = evaluate_retrieval(retrieval_results, qrels_lookup, k=k)
    row = {'model': name}
    row.update(metrics)
    row['queries_evaluated'] = len(retrieval_results)
    return row


results_table = []

# TF-IDF
tfidf_results = run_tfidf_search(docs_eval, queries_eval, top_k=EVAL_K)
qids_eval = queries_eval['id'].astype(str).tolist()
qrels_subset = {qid: qrels.get(qid, []) for qid in qids_eval}
results_table.append(evaluate_model('TF-IDF', tfidf_results, qrels_subset, EVAL_K))

# BM25+
bm25_results = run_bm25_search(docs_eval, queries_eval, top_k=EVAL_K)
results_table.append(evaluate_model('BM25+', bm25_results, qrels_subset, EVAL_K))

# Dense (embedding-based)
if RUN_DENSE:
    dense_queries = queries_eval.head(DENSE_QUERY_LIMIT).copy()
    dense_results = run_dense_search(docs_eval, dense_queries, top_k=EVAL_K)
    dense_qids = dense_queries['id'].astype(str).tolist()
    dense_qrels_subset = {qid: qrels.get(qid, []) for qid in dense_qids}
    results_table.append(evaluate_model('Dense Embedding', dense_results, dense_qrels_subset, EVAL_K))
else:
    results_table.append({
        'model': 'Dense Embedding (skipped)',
        'avg_recall': float('nan'),
        'avg_precision': float('nan'),
        'mrr': float('nan'),
        'map': float('nan'),
        'queries_evaluated': 0,
    })

comparison_df = pd.DataFrame(results_table)
comparison_df


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,model,avg_recall,avg_precision,mrr,map,queries_evaluated
0,TF-IDF,0.019274,0.0130,0.064056,0.010993,200
1,BM25+,0.020844,0.0145,0.084722,0.014518,200
2,Dense Embedding,0.029229,0.0180,0.110000,0.022729,50


In [4]:
# Optional: compare the top documents for one query across models.
example_idx = 0
query_id = str(queries_eval.iloc[example_idx]['id'])
query_text = queries_eval.iloc[example_idx]['content']

print('query_id:', query_id)
print('query_text:', query_text[:300], '...')

print('\nTF-IDF top docs:')
print(tfidf_results[example_idx]['relevant_docs'][:5])

print('\nBM25+ top docs:')
print(bm25_results[example_idx]['relevant_docs'][:5])

if RUN_DENSE:
    print('\nDense top docs:')
    print(dense_results[0]['relevant_docs'][:5])


query_id: 961c4349-8cf1-4ef1-89cc-24d20bb9d000_67878
query_text: want to try reformatting damaged sd card ...

TF-IDF top docs:
['9b7f0646-d400-4843-b5d8-df4ce4cd7b27_7240', '135f5fdb-bcba-40b8-b90d-823617f1e805_21141', '14bd0cda-cd52-4771-a2b0-d9383455e6e6_44400', '2bb3863e-ec0b-4096-a857-d3dc77aada38_34510', '8c5c53dd-d1b0-49ea-acf8-67a7fc90fe62_49657']

BM25+ top docs:
['9b7f0646-d400-4843-b5d8-df4ce4cd7b27_7240', '135f5fdb-bcba-40b8-b90d-823617f1e805_21141', '8aea6e52-c8c2-41db-a547-1396c4ba7334_54792', '14bd0cda-cd52-4771-a2b0-d9383455e6e6_44400', '6a5ceecb-6ade-47bb-96dc-014e80f185d2_52521']

Dense top docs:
['9b7f0646-d400-4843-b5d8-df4ce4cd7b27_7240', '135f5fdb-bcba-40b8-b90d-823617f1e805_21141', '8498c8d3-0c6b-465f-9035-0833cf57f2e6_6378', '8aea6e52-c8c2-41db-a547-1396c4ba7334_54792', '2bb3863e-ec0b-4096-a857-d3dc77aada38_34510']


## Interpretation

- **TF-IDF** and **BM25+** are lexical baselines.
- **Dense embedding retrieval** captures semantic similarity beyond exact token overlap.
- The metric table above is the Phase-1 baseline comparison.
